# 🧠 Glioma Segmentation — 2D U-Net MULTICLASE + BraTS2020
**3 clases:** edema peritumoral | núcleo necrótico | tumor activo

---
### ✅ Antes de correr
1. **Runtime → Change runtime type → T4 GPU**
2. Ejecuta las celdas en orden

### 📁 Drive
```
Mi unidad/brats2020_unet/
  best_unet2d_multiclass.keras   ← modelo multiclase (nuevo)
  best_unet2d.keras              ← modelo binario (se conserva)
  history_multiclass.npy
  test_files_multiclass.npy
  learning_curves_multiclass.png
  qualitative_multiclass.png
```


## 📦 CELDA 1 — Instalar dependencias

In [ ]:
!pip install -q kaggle h5py scikit-learn scipy matplotlib numpy

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 💾 CELDA 2 — Montar Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/brats2020_unet'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'Drive montado ✓')
print(f'\nContenido Drive:')
archivos = sorted(os.listdir(DRIVE_DIR))
if len(archivos) == 0:
    print('  (vacío)')
else:
    for f in archivos:
        fpath = os.path.join(DRIVE_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1e6
            print(f'  {f:<45} {size:>8.1f} MB')

## ⚙️ CELDA 3 — Configuración global

In [ ]:
import os
import glob
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import directed_hausdorff

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation,
    MaxPooling2D, Conv2DTranspose, concatenate, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
)

# ─── Rutas Drive ──────────────────────────────────────────────────
DRIVE_DIR      = '/content/drive/MyDrive/brats2020_unet'
MODEL_PATH     = os.path.join(DRIVE_DIR, 'best_unet2d_multiclass.keras')
HISTORY_PATH   = os.path.join(DRIVE_DIR, 'history_multiclass.npy')
TEST_PATH      = os.path.join(DRIVE_DIR, 'test_files_multiclass.npy')
CURVES_PATH    = os.path.join(DRIVE_DIR, 'learning_curves_multiclass.png')
QUAL_PATH      = os.path.join(DRIVE_DIR, 'qualitative_multiclass.png')

# ─── Hiperparámetros ──────────────────────────────────────────────
DATA_DIR    = '/content/brats2020_h5'
IMG_SIZE    = 128
N_CHANNELS  = 4      # T1, T1ce, T2, FLAIR
N_CLASSES   = 3      # edema | núcleo necrótico | tumor activo
BATCH_SIZE  = 16
EPOCHS      = 15
LR          = 1e-3
SEED        = 42
THRESHOLD   = 0.5

# Colores por clase (igual que el paper)
CLASS_COLORS = [
    (0.85, 0.50, 0.50),   # canal 0 → edema peritumoral  → rosado
    (0.15, 0.25, 0.85),   # canal 1 → núcleo necrótico   → azul
    (0.90, 0.08, 0.08),   # canal 2 → tumor activo       → rojo
]
CLASS_NAMES = ['Edema', 'Necrotic Core', 'Enhancing Tumor']

print('Configuración cargada ✓')
print(f'  N_CLASSES={N_CLASSES} | BATCH_SIZE={BATCH_SIZE} | EPOCHS={EPOCHS}')

## 📥 CELDA 4 — Descargar dataset
> ⚠️ Salta si el dataset ya está en `/content/brats2020_h5`

In [ ]:
import os, glob

h5_check = glob.glob('/content/brats2020_h5/**/*.h5', recursive=True)

if len(h5_check) > 0:
    print(f'Dataset ya existe → {len(h5_check)} archivos .h5')
else:
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('Sube tu kaggle.json:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)

    os.makedirs('/content/brats2020_h5', exist_ok=True)
    print('Descargando dataset (~7 GB)...')
    !kaggle datasets download -d awsaf49/brats2020-training-data \
        -p /content/brats2020_h5
    print('Descomprimiendo...')
    !unzip -q /content/brats2020_h5/brats2020-training-data.zip \
        -d /content/brats2020_h5
    !rm /content/brats2020_h5/brats2020-training-data.zip
    h5_final = glob.glob('/content/brats2020_h5/**/*.h5', recursive=True)
    print(f'✓ Archivos .h5: {len(h5_final)}')

## 📂 CELDA 5 — Lectura multiclase + Generador + Split

In [ ]:
# ─── Lista de archivos ────────────────────────────────────────────
all_files = sorted(glob.glob(
    os.path.join(DATA_DIR, '**', '*.h5'), recursive=True
))
if len(all_files) == 0:
    raise RuntimeError('No se encontraron archivos .h5 — corre Celda 4')
print(f'Total archivos .h5: {len(all_files)}')


# ─── Leer muestra H5 — máscara con 3 canales originales ──────────
def read_h5_multiclass(path, img_size=128):
    """
    Retorna:
      x: (img_size, img_size, 4)  — 4 modalidades
      y: (img_size, img_size, 3)  — 3 clases sin binarizar juntas
         canal 0 = edema peritumoral
         canal 1 = núcleo necrótico
         canal 2 = tumor activo (enhancing)
    """
    with h5py.File(path, 'r') as f:
        x = f['image'][()].astype(np.float32)   # (H, W, 4)
        y = f['mask'][()].astype(np.float32)    # (H, W, 3)

    # Si la máscara es binaria (1 canal) expandir a 3
    if y.ndim == 2:
        y = np.stack([y, y, y], axis=-1)
    elif y.ndim == 3 and y.shape[-1] == 1:
        y = np.concatenate([y, y, y], axis=-1)

    x = tf.image.resize(x, [img_size, img_size], method='bilinear').numpy()
    y = tf.image.resize(y, [img_size, img_size], method='nearest').numpy()
    y = (y > 0).astype(np.float32)   # binarizar cada canal por separado
    return x, y   # y: (128,128,3)


# ─── Generador lazy-loading multiclase ───────────────────────────
class MulticlassH5Generator(tf.keras.utils.Sequence):
    def __init__(self, file_list, batch_size=16,
                 shuffle=True, img_size=128):
        self.file_list  = np.array(file_list)
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.img_size   = img_size
        self.indices    = np.arange(len(self.file_list))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.file_list) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.file_list[
            self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        ]
        X, Y = [], []
        for path in batch:
            x, y = read_h5_multiclass(path, self.img_size)
            X.append(x)
            Y.append(y)
        return np.array(X, np.float32), np.array(Y, np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ─── Split ────────────────────────────────────────────────────────
train_files, test_files = train_test_split(
    all_files, test_size=0.10, random_state=SEED
)
train_files, val_files = train_test_split(
    train_files, test_size=0.15/0.90, random_state=SEED
)
np.save(TEST_PATH, np.array(test_files))
print(f'Train:{len(train_files)} | Val:{len(val_files)} | Test:{len(test_files)}')

train_gen = MulticlassH5Generator(train_files, BATCH_SIZE, shuffle=True,  img_size=IMG_SIZE)
val_gen   = MulticlassH5Generator(val_files,   BATCH_SIZE, shuffle=False, img_size=IMG_SIZE)
print(f'Pasos/época → train:{len(train_gen)} | val:{len(val_gen)}')

# Verificar
Xb, yb = train_gen[0]
print(f'Batch imagen : {Xb.shape}')
print(f'Batch máscara: {yb.shape}  ← debe ser (16,128,128,3)')

## 🏗️ CELDA 6 — U-Net Multiclase (3 salidas)

In [ ]:
# ─── Dice multiclase (promedio de las 3 clases) ───────────────────
def multiclass_dice(y_true, y_pred, smooth=1e-6):
    """
    Dice promedio sobre las 3 clases.
    y_true, y_pred: (batch, H, W, 3)
    """
    dice_sum = 0.0
    for c in range(N_CLASSES):
        yt = tf.keras.backend.flatten(
            tf.cast(y_true[..., c], tf.float32))
        yp = tf.keras.backend.flatten(
            tf.cast(y_pred[..., c], tf.float32))
        intersection = tf.reduce_sum(yt * yp)
        dice_sum += (2.0*intersection+smooth) / (
            tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)
    return dice_sum / N_CLASSES

def multiclass_dice_loss(y_true, y_pred):
    return 1.0 - multiclass_dice(y_true, y_pred)


# ─── Bloques convolucionales ──────────────────────────────────────
def conv_block(x, filters, dropout_rate=0.1):
    x = Conv2D(filters, 3, padding='same',
               kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, padding='same',
               kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    return x


# ─── U-Net con 3 salidas (sigmoid por canal) ──────────────────────
def build_unet_multiclass(img_size=128, n_channels=4, n_classes=3):
    """
    Igual arquitectura que la binaria pero con:
    - Output: n_classes canales con activación sigmoid
    - Cada canal predice una clase independientemente
    """
    inp = Input(shape=(img_size, img_size, n_channels))

    # Encoder
    c1 = conv_block(inp, 32,  0.1);  p1 = MaxPooling2D(2)(c1)
    c2 = conv_block(p1,  64,  0.1);  p2 = MaxPooling2D(2)(c2)
    c3 = conv_block(p2,  128, 0.2);  p3 = MaxPooling2D(2)(c3)
    c4 = conv_block(p3,  256, 0.2);  p4 = MaxPooling2D(2)(c4)

    # Bottleneck
    bn = conv_block(p4, 512, 0.3)

    # Decoder
    u6 = Conv2DTranspose(256, 2, strides=2, padding='same')(bn)
    c6 = conv_block(concatenate([u6, c4]), 256, 0.2)
    u7 = Conv2DTranspose(128, 2, strides=2, padding='same')(c6)
    c7 = conv_block(concatenate([u7, c3]), 128, 0.2)
    u8 = Conv2DTranspose(64,  2, strides=2, padding='same')(c7)
    c8 = conv_block(concatenate([u8, c2]), 64,  0.1)
    u9 = Conv2DTranspose(32,  2, strides=2, padding='same')(c8)
    c9 = conv_block(concatenate([u9, c1]), 32,  0.1)

    # Output: 3 canales, sigmoid independiente por clase
    out = Conv2D(n_classes, 1, activation='sigmoid',
                 name='output_multiclass')(c9)

    return Model(inp, out, name='UNet2D_Multiclass')


# ─── Cargar si existe, si no construir ────────────────────────────
if os.path.exists(MODEL_PATH):
    print(f'Modelo multiclase encontrado en Drive → cargando...')
    model = tf.keras.models.load_model(
        MODEL_PATH,
        custom_objects={
            'multiclass_dice_loss': multiclass_dice_loss,
            'multiclass_dice':      multiclass_dice
        }
    )
    print('Modelo cargado ✓ — puedes saltar a Celda 9')
else:
    model = build_unet_multiclass(IMG_SIZE, N_CHANNELS, N_CLASSES)
    model.compile(
        optimizer=Adam(LR),
        loss=multiclass_dice_loss,
        metrics=[multiclass_dice,
                 tf.keras.metrics.Recall(name='sensitivity')]
    )
    print('Modelo multiclase construido ✓')

model.summary()
print(f'\nParámetros entrenables: {model.count_params():,}')
print(f'Shape salida: {model.output_shape}  ← debe ser (None,128,128,3)')

## 🚀 CELDA 7 — Entrenamiento multiclase
> Tiempo estimado en T4: ~5-6 horas (15 épocas)

In [ ]:
if os.path.exists(HISTORY_PATH):
    print('Historial ya existe en Drive.')
    print('Para re-entrenar borra history_multiclass.npy de Drive.')
    history_dict = np.load(HISTORY_PATH, allow_pickle=True).item()
else:
    callbacks = [
        ModelCheckpoint(
            MODEL_PATH,
            monitor='val_multiclass_dice', mode='max',
            save_best_only=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1
        ),
        EarlyStopping(
            monitor='val_loss', patience=7,
            restore_best_weights=True, verbose=1
        ),
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    history_dict = history.history
    np.save(HISTORY_PATH, history_dict)
    print(f'\n✓ Modelo guardado   → {MODEL_PATH}')
    print(f'✓ Historial guardado → {HISTORY_PATH}')

## 📈 CELDA 8 — Curvas de aprendizaje (estilo paper)

In [ ]:
epochs = list(range(1, len(history_dict['loss']) + 1))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epochs, history_dict['loss'],
        color='blue', linewidth=2,
        marker='o', markersize=6,
        label='Training Loss (Dice)')
ax.plot(epochs, history_dict['val_loss'],
        color='red', linewidth=2,
        linestyle='--', marker='s', markersize=6,
        label='Validation Loss (Dice)')

ax.set_title('U-Net Multiclass Model Convergence',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Epochs', fontsize=13)
ax.set_ylabel('Dice Loss', fontsize=13)
ax.set_xlim(0.5, len(epochs) + 0.5)
ax.set_xticks(epochs)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Figura guardada → {CURVES_PATH}')

## 📊 CELDA 9 — Evaluación cuantitativa (métricas como el paper)
> DSC global, Sensitivity, Specificity, Hausdorff

In [ ]:
test_files_eval = np.load(TEST_PATH, allow_pickle=True).tolist()
print(f'Evaluando {len(test_files_eval)} muestras...')

dices_global = []
dices_per_class = [[], [], []]   # edema, núcleo, tumor activo
senss, specs, hds = [], [], []

for path in test_files_eval:
    x, y_true = read_h5_multiclass(str(path), IMG_SIZE)
    y_prob = model.predict(x[np.newaxis,...], verbose=0)[0]   # (128,128,3)
    y_pred = (y_prob >= THRESHOLD).astype(np.uint8)

    # ── Métricas globales (unión de las 3 clases) ────────────────
    yt_bin = (y_true.sum(axis=-1) > 0).astype(np.uint8).ravel()
    yp_bin = (y_pred.sum(axis=-1) > 0).astype(np.uint8).ravel()

    TP = np.sum((yp_bin==1)&(yt_bin==1))
    FP = np.sum((yp_bin==1)&(yt_bin==0))
    TN = np.sum((yp_bin==0)&(yt_bin==0))
    FN = np.sum((yp_bin==0)&(yt_bin==1))
    e  = 1e-6

    dices_global.append((2*TP+e)/(2*TP+FP+FN+e))
    senss.append((TP+e)/(TP+FN+e))
    specs.append((TN+e)/(TN+FP+e))

    pts_t = np.argwhere(yt_bin.reshape(IMG_SIZE,IMG_SIZE)>0)
    pts_p = np.argwhere(yp_bin.reshape(IMG_SIZE,IMG_SIZE)>0)
    hd = np.nan if len(pts_t)==0 or len(pts_p)==0 else max(
        directed_hausdorff(pts_t,pts_p)[0],
        directed_hausdorff(pts_p,pts_t)[0]
    )
    if not np.isnan(hd): hds.append(hd)

    # ── DSC por clase ─────────────────────────────────────────────
    for c in range(N_CLASSES):
        yt_c = y_true[...,c].ravel().astype(np.uint8)
        yp_c = y_pred[...,c].ravel().astype(np.uint8)
        tp_c = np.sum((yp_c==1)&(yt_c==1))
        fp_c = np.sum((yp_c==1)&(yt_c==0))
        fn_c = np.sum((yp_c==0)&(yt_c==1))
        dices_per_class[c].append((2*tp_c+e)/(2*tp_c+fp_c+fn_c+e))

print('\n' + '='*56)
print('  RESULTADOS TEST SET  (métricas globales como el paper)')
print('='*56)
print(f'  DSC          {np.mean(dices_global):.4f}   (paper binario: 0.901)')
print(f'  Sensitivity  {np.mean(senss):.4f}   (paper binario: 0.909)')
print(f'  Specificity  {np.mean(specs):.4f}   (paper binario: 0.999)')
print(f'  Hausdorff    {np.mean(hds):.2f} mm (paper binario: 2.11 mm)')
print('='*56)
print('\n  DSC por clase:')
for c, name in enumerate(CLASS_NAMES):
    print(f'    {name:<20} {np.mean(dices_per_class[c]):.4f}')
print('='*56)

## 🖼️ CELDA 10 — Figura multiclase estilo paper (Fig. 4)
Fondo blanco · 3 filas · colores: rosado=edema | azul=núcleo | rojo=tumor activo

In [ ]:
def normalize_display(img):
    mn, mx = img.min(), img.max()
    return img if mx-mn < 1e-8 else (img-mn)/(mx-mn)

def overlay_multiclass(base_gray, mask3ch, alpha=0.55):
    """
    Superpone máscara multiclase sobre imagen en gris.
    Orden de pintado: edema primero (fondo), tumor activo último (encima)
    """
    rgb = np.stack([base_gray]*3, axis=-1).copy()
    for c, color in enumerate(CLASS_COLORS):
        m = mask3ch[:,:,c].astype(bool)
        if m.sum() == 0:
            continue
        rgb[m] = (1-alpha)*rgb[m] + alpha*np.array(color)
    return np.clip(rgb, 0, 1)


# Seleccionar 3 muestras con al menos 2 clases visibles
N_VIS  = 3
chosen = []
for path in test_files_eval:
    x, y = read_h5_multiclass(str(path), IMG_SIZE)
    clases = (y.sum(axis=(0,1)) > 50).sum()
    if clases >= 2:
        chosen.append((x, y))
    if len(chosen) == N_VIS:
        break

print(f'Muestras seleccionadas: {len(chosen)}')

fig, axes = plt.subplots(N_VIS, 3,
                          figsize=(8, 3.2*N_VIS),
                          facecolor='white')

col_titles = ['FLAIR', 'Ground\nTruth', 'Axial\n(Prediction)']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11,
                            fontweight='bold', pad=8,
                            color='black')

for row, (x, y) in enumerate(chosen):
    yp     = model.predict(x[np.newaxis,...], verbose=0)[0]  # (128,128,3)
    yp_bin = (yp >= THRESHOLD).astype(np.float32)
    flair  = normalize_display(x[:,:,0])

    imgs  = [
        flair,
        overlay_multiclass(flair, y),       # GT con 3 colores
        overlay_multiclass(flair, yp_bin),  # Predicción con 3 colores
    ]
    cmaps = ['gray', None, None]

    for col in range(3):
        ax = axes[row, col]
        ax.imshow(imgs[col], cmap=cmaps[col], vmin=0, vmax=1)
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.2)
        ax.set_xticks([])
        ax.set_yticks([])

for col, label in enumerate(['(a)', '(b)', '(c)']):
    axes[N_VIS-1, col].set_xlabel(label, fontsize=12,
                                   fontweight='bold',
                                   color='black', labelpad=6)

plt.subplots_adjust(wspace=0.05, hspace=0.08,
                    left=0.02, right=0.98,
                    top=0.93, bottom=0.05)
plt.savefig(QUAL_PATH, dpi=180, facecolor='white', bbox_inches='tight')
plt.show()
print(f'✓ Figura guardada → {QUAL_PATH}')
print('\nLeyenda:')
print('  Rosado → edema peritumoral')
print('  Azul   → núcleo necrótico')
print('  Rojo   → tumor activo (enhancing)')

## ✅ CELDA 11 — Verificar Drive

In [ ]:
print(f'Archivos en Drive → {DRIVE_DIR}/')
for fname in sorted(os.listdir(DRIVE_DIR)):
    fpath = os.path.join(DRIVE_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {fname:<45} {size:>8.1f} MB')
print('\n✓ Experimento multiclase completo.')